# 📊 Model Monitoring
This notebook demonstrates various techniques for monitoring machine learning models in production, with a focus on fraud detection systems.

## 🛠️ Setup Environment <a name="setup"></a>

In [6]:
%reload_ext autoreload
%autoreload 2

## 📚 Import Libraries

In [63]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import shap

import pickle

import sys
sys.path.append('../../')


## 📥 Load Data and Model

In [ ]:
score_balanced_model = pickle.load(open('../../models/score_balanced_model.pkl', 'rb'))

In [9]:
# Load training data
X = pd.read_parquet('../../data/train_data.parquet')
metadata_columns = ['trans_date_trans_time', 'gender', 'street']
train_y = X['is_fraud']
train_X = X.drop(columns=['is_fraud'] + metadata_columns)

# Load holdout and OOT data
holdout = pd.read_parquet('../../data/holdout_data.parquet')
holdout_y = holdout['is_fraud']
holdout_X = holdout.drop(columns=['is_fraud'] + metadata_columns)

oot = pd.read_parquet('../../data/oot_data.parquet')
oot_y = oot['is_fraud']
oot_X = oot.drop(columns=['is_fraud'] + metadata_columns)

 ## 📈 Model Monitoring Metrics

Standard evaluation metrics may not be sufficient for fraud detection models due to incomplete confusion matrices. Here we demonstrate various business and technical KPIs for monitoring model performance:

- Population Stability Index (PSI)
- Feature importance drift
- Distribution drift
- Business metrics drift

I will use oot dataset as example dataset. In real world applications this will be the live dataset.

Disclamer: This is a non exausting demo. Try your own metrics and have fun!

## 🔄 Model Drift Analysis
Population Stability Index (PSI) is used to measure score distribution drift over time. 

Interpretation guidelines:
- PSI < 0.1: No significant change, model can continue to be used
- 0.1 ≤ PSI < 0.2: Minor changes detected, consider monitoring closely
- PSI ≥ 0.2: Significant changes detected, model retraining recommended

use PSI to calculate drift in score distribution

In [28]:
import numpy as np

def calculate_drift(expected: np.ndarray, actual: np.ndarray, bucket_type: str = "bins", n_bins: int = 10) -> float:
    """Calculate PSI metric for two arrays.

    Parameters
    ----------
        expected : list-like
            Array of expected values    
        actual : list-like
            Array of actual values
        bucket_type : str
            Binning strategy. Accepts two options: 'bins' and 'quantiles'. Defaults to 'bins'.
            'bins': input arrays are splitted into bins with equal
                and fixed steps based on 'expected' array
            'quantiles': input arrays are binned according to 'expected' array
                with given number of n_bins
        n_bins : int
            Number of buckets for binning. Defaults to 10.

    Returns
    -------
        A single float number
    """
    psi = _calculate_psi(expected, actual, bucket_type, n_bins)
    drift = psi >= 0.2

    return {"psi": psi, "drift_status": drift}



def _psi(expected: np.ndarray, actual: np.ndarray, bucket_type: str = "bins", n_bins: int = 10) -> float:
    """Calculate PSI metric for two arrays.
    
    Parameters
    ----------
        expected : list-like
            Array of expected values
        actual : list-like
            Array of actual values
        bucket_type : str
            Binning strategy. Accepts two options: 'bins' and 'quantiles'. Defaults to 'bins'.
            'bins': input arrays are splitted into bins with equal
                and fixed steps based on 'expected' array
            'quantiles': input arrays are binned according to 'expected' array
                with given number of n_bins
        n_bins : int
            Number of buckets for binning. Defaults to 10.

    Returns
    -------
        A single float number
    """
    breakpoints = np.arange(0, n_bins + 1) / (n_bins) * 100
    if bucket_type == "bins":
        breakpoints = np.histogram(expected, n_bins)[1]
    elif bucket_type == "quantiles":
        breakpoints = np.percentile(expected, breakpoints)

    # Calculate frequencies
    expected_percents = np.histogram(expected, breakpoints)[0] / len(expected)
    actual_percents = np.histogram(actual, breakpoints)[0] / len(actual)
    # Clip freaquencies to avoid zero division
    expected_percents = np.clip(expected_percents, a_min=0.0001, a_max=None)
    actual_percents = np.clip(actual_percents, a_min=0.0001, a_max=None)
    # Calculate PSI
    psi_value = (expected_percents - actual_percents) * np.log(expected_percents / actual_percents)
    psi_value = sum(psi_value)

    return psi_value


def _calculate_psi(
        expected: np.ndarray, actual: np.ndarray, bucket_type: str = "bins", n_bins: int = 10, axis: int = 0
) -> np.ndarray:
    """Apply PSI calculation to 2 1-d or 2-d arrays.

    Parameters
    ----------
    expected : list-like
        Array of expected values
    actual : list-like
        Array of actual values
    bucket_type : str
        Binning strategy. Accepts two options: 'bins' and 'quantiles'. Defaults to 'bins'.
            'bins' - input arrays are splitted into bins with equal
                and fixed steps based on ’expected' array
            'quantiles' - input arrays are binned according to ’expected’ array
                with given number of n_bins
    n_bins : int
        Number of buckets for binning. Defaults to 10.

    Returns
    -------
        np.ndarray
    """
    if len(expected.shape) == 1:
        psi_values = np.empty(len(expected.shape))
    else:
        psi_values = np.empty(expected.shape[axis])

    for i in range(0, len(psi_values)):
        if len(psi_values) == 1:
            psi_values = _psi(expected, actual, bucket_type, n_bins)
        elif axis == 0:
            psi_values[i] = _psi(expected[:, i], actual[:, i], bucket_type, n_bins)
        elif axis == 1:
            psi_values[i] = _psi(expected[i, :], actual[i, :], bucket_type, n_bins)
        return np.array(psi_values)
    

In [ ]:
y_pred_train = score_balanced_model.predict_proba(train_X)[:,1] 
y_pred_oot = score_balanced_model.predict_proba(oot_X)[:,1] 
psi_drift = calculate_drift(y_pred_train,y_pred_oot)
psi_drift


## 🔍 Data Drift Detection
Monitor changes in feature distributions and importance over time. We'll analyze:
- Feature importance drift using SHAP values
- Distribution drift using statistical tests
- Categorical variable drift using Chi-Square tests

### Feature Importance Drift
Use SHAP values to analyze changes in feature importance between training and obsevation datasets. This helps identify:
- Which features are becoming more/less important over time
- Potential shifts in feature relationships
- Model behavior changes in production
- Stability of feature contributions to predictions

SHAP values provide a unified measure of feature importance that:
- Considers both the magnitude and direction of feature impacts
- Maintains consistency across different model types
- Helps explain individual predictions and global patterns
- Enables comparison of feature importance across different time periods

In [ ]:
shap_values_train = shap.TreeExplainer(score_balanced_model.best_estimator_['class']).shap_values(train_X)
shap_values_oot = shap.TreeExplainer(score_balanced_model.best_estimator_['class']).shap_values(oot_X)


In [ ]:
shap.summary_plot(shap_values_train, train_X)

In [ ]:
shap.summary_plot(shap_values_oot, oot_X)

In [126]:
feature_importance = pd.DataFrame(index=train_X.columns.values)

In [127]:
df_shap_values_train = pd.DataFrame(shap_values_train)
df_shap_values_train.columns = train_X.columns.values
feature_importance["train"] = df_shap_values_train.abs().mean()   
feature_importance["train_sorted"] = feature_importance["train"].sort_values(ascending=False).reset_index().index+1

In [128]:
df_shap_values_oot = pd.DataFrame(shap_values_oot)
df_shap_values_oot.columns = train_X.columns.values
feature_importance["oot"] = df_shap_values_oot.abs().mean()   
feature_importance["oot_sorted"] = feature_importance["oot"].sort_values(ascending=False).reset_index().index+1

In [ ]:
feature_importance["drift"] = abs(feature_importance["oot_sorted"]-feature_importance["train_sorted"])
feature_importance.sort_values(by="drift", ascending=False)

### Variable Distribution Drift
For numerical variables, we use the Kolmogorov-Smirnov test:
- Non-parametric test (no assumptions about data distribution)
- Tests equality of continuous probability distributions
- Sensitive to differences in both location and shape of distributions
- Particularly useful for comparing two samples

The test helps answer:
- How likely are these samples from the same distribution?
- What is the magnitude of distribution differences?
- Which features show significant drift?

In [4]:
from scipy.stats import ks_2samp

def detect_covariate_drift(base_df,current_df,threshold=0.05)->bool:
  status = True
  report={}
  for column in base_df.columns:
    d1 = base_df[column]
    d2 = current_df[column]
    is_same_dist = ks_2samp(d1,d2)

    if threshold<=is_same_dist.pvalue:
      is_found=False
    else:
      status = False
      is_found=True

    report.update({column:{
    "p_value":float(is_same_dist.pvalue),
    "drift_status":is_found}}) 
    
  return report


In [ ]:
covariate_drift = detect_covariate_drift(train_X,oot_X)
covariate_drift = pd.DataFrame(covariate_drift).T
features_with_drift = covariate_drift[covariate_drift['drift_status']==True]
features_with_drift

In [ ]:
#ADD VIZ HERE on true drifts
f = features_with_drift.index.tolist()[0]
# Plotting the KDE Plot
sns.kdeplot(train_X[f], color='r', fill=True, label='Base_Population')
sns.kdeplot(oot_X[f], color='b', fill=True, label='Test_Population')
plt.xlabel(f)
plt.ylabel('Probability Density Function')
plt.legend()
plt.plot()


### Categorical Variables Drift
For categorical variables, we use the Chi-Square Test to detect changes in category distributions. This helps identify:
- Changes in category frequencies
- Shifts in categorical feature importance
- Potential data quality issues

## 📊 Population Drift
Monitor business metrics to ensure model performance aligns with business objectives:
- Fraud rate changes over time
- Conversion rate variations
- Transaction volume trends
- False positive/negative rates
- Cost of fraud vs. cost of intervention